# 07 - What the recorded readout noise looks like

Before modelling or classifying readout traces, it helps to know their noise. This notebook measures the recorded dataset
(the transmon readout of Di Guglielmo et al.: 770 complex samples at 3.25 ns, 2.5 µs readout pulse, $T_1$ = 32 µs) and
answers the questions that decide how to model it (paper Sec. V):

1. Is the noise the same for both states?
2. Is it stationary along the trace?
3. Is it white, or correlated from sample to sample?
4. Is it Gaussian?
5. Is it the same in I and Q?
6. What is the best a linear classifier can do (the matched-filter ceiling)?
7. Is qubit decay during the window visible in the data?

**You need:** `0528_X_train_0_770.npy` and `0528_y_train_0_770.npy` from the dataset (see `DATA_MANIFEST.csv`).

In [ ]:
from pathlib import Path

DATA_DIR = Path("..")
BLOCK = 5000          # the file alternates class in runs of 5,000 shots
N_BLOCKS = 16         # blocks sampled, spread across the whole file so slow drift does not look like noise
PER_BLOCK = 1500      # shots taken from each sampled block
DT_NS = 3.25          # recording's sample spacing

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy.special import ndtr

X = np.load(DATA_DIR / "0528_X_train_0_770.npy", mmap_mode="r")
y = np.load(DATA_DIR / "0528_y_train_0_770.npy").astype(int)
idx0, idx1 = [], []
for b in np.linspace(0, X.shape[0] // BLOCK - 2, N_BLOCKS).astype(int):
    lo = b * BLOCK
    (idx0 if y[lo] == 0 else idx1).append(np.arange(lo, lo + PER_BLOCK))
A0, A1 = (np.asarray(X[np.concatenate(i)]).astype(np.float64) for i in (idx0, idx1))
z0, z1 = A0[:, 0::2] + 1j * A0[:, 1::2], A1[:, 0::2] + 1j * A1[:, 1::2]
del A0, A1
m0, m1 = z0.mean(0), z1.mean(0)
r0, r1 = z0 - m0, z1 - m1
T = z0.shape[1]
print(f"{len(z0)} |0> and {len(z1)} |1> shots, {T} samples each")

## 1–2. State dependence and stationarity

Additive noise is the same for both states by construction. If the excited state were noisier (from decay during the
window), an additive model would be wrong in structure, not just in scale.

In [ ]:
print(f"residual std |0> {np.std(r0):.1f}, |1> {np.std(r1):.1f}, ratio {np.std(r1) / np.std(r0):.4f}")
print(f"mean class separation |m1 - m0| = {np.mean(np.abs(m1 - m0)):.1f}  (per-sample SNR ~ {np.mean(np.abs(m1 - m0)) / np.std(r0):.3f})")
st0, st1 = np.std(r0, axis=0), np.std(r1, axis=0)
print(f"sigma_0 along the trace: min {st0.min():.1f}, max {st0.max():.1f} (spread {100 * (st0.max() / st0.min() - 1):.1f}%)")

t = np.arange(T) * DT_NS / 1000
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(t, np.abs(m0), label="|0⟩"); ax[0].plot(t, np.abs(m1), label="|1⟩")
ax[0].set_title("class-mean trajectories |m|"); ax[0].set_xlabel("time (us)"); ax[0].legend()
ax[1].plot(t, st0, label="|0⟩"); ax[1].plot(t, st1, label="|1⟩")
ax[1].set_title("noise std per sample"); ax[1].set_xlabel("time (us)"); ax[1].legend()
plt.tight_layout(); plt.show()

## 3. Colour

Correlation between samples sets how fast integration helps: the gain is $\sqrt{N/\rho^*}$, not $\sqrt N$. The power
spectrum shows the readout chain's anti-alias and decimation filtering. Its DC bin is raised, which is a per-shot offset
that does not average down.

In [ ]:
rn = r0 / np.std(r0)
lags = np.arange(1, 51)
ac = np.array([np.mean(np.real(rn[:, :-k] * np.conj(rn[:, k:]))) for k in lags])
print("autocorrelation, lags 1-5:", np.round(ac[:5], 4))
rho_star = 1 + 2 * np.sum((1 - lags / T) * ac)
print(f"effective integration gain sqrt(N/rho*) = {np.sqrt(T / rho_star):.2f}, vs sqrt(N) = {np.sqrt(T):.2f}")

psd = (np.abs(np.fft.fft(r0, axis=1)) ** 2).mean(0); psd /= psd.mean()
f = np.fft.fftshift(np.fft.fftfreq(T))
print(f"normalised PSD at DC {psd[0]:.3f}; predicted per-shot offset std sigma*sqrt(PSD[0]/N) = "
      f"{np.std(r0) * np.sqrt(psd[0] / T):.1f}, measured {np.std(r0.mean(1)):.1f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].stem(lags[:15], ac[:15]); ax[0].set_xlabel("lag (samples)"); ax[0].set_title("autocorrelation |0⟩")
ax[1].semilogy(f, np.fft.fftshift(psd)); ax[1].set_xlabel("frequency (cycles/sample)"); ax[1].set_title("normalised noise PSD")
plt.tight_layout(); plt.show()

## 4–5. Gaussianity and I/Q symmetry

Heavy tails (large excess kurtosis) would mean jump events; I/Q asymmetry would mean the noise depends on the phase.

In [ ]:
for lab, r in (("|0>", r0), ("|1>", r1)):
    v = r.real.ravel() / np.std(r.real)
    v = v[::max(1, len(v) // 200000)]
    rho = np.corrcoef(r.real.ravel()[::97], r.imag.ravel()[::97])[0, 1]
    print(f"{lab}: skew {stats.skew(v):+.4f}, excess kurtosis {stats.kurtosis(v):+.4f}, "
          f"std_I/std_Q {np.std(r.real) / np.std(r.imag):.4f}, corr(I,Q) {rho:+.4f}")

## 6. The matched-filter ceiling

Fit $w = m_1 - m_0$ on one half of each class, score the other half. Compare the measured fidelity with what Gaussian
classes at the same separation would give, $\Phi(d/2)$.

In [ ]:
h0, h1 = len(z0) // 2, len(z1) // 2
w = z1[:h1].mean(0) - z0[:h0].mean(0)
ev0, ev1 = z0[h0:], z1[h1:]
mf = lambda z, sl=slice(None): (z[:, sl] * np.conj(w[sl])).real.sum(1)
s0, s1 = mf(ev0), mf(ev1)
d = abs(s1.mean() - s0.mean()) / np.sqrt(0.5 * (s0.var() + s1.var()))
thr = 0.5 * (s0.mean() + s1.mean())
F = 0.5 * (np.mean(s0 < thr) + np.mean(s1 > thr)) if s1.mean() > s0.mean() else 0.5 * (np.mean(s0 > thr) + np.mean(s1 < thr))
print(f"d = {d:.3f} sigma: Gaussian prediction {100 * ndtr(d / 2):.2f}%, measured {100 * F:.2f}%")
print("The shortfall is the non-Gaussian part: shots that decay or start in the wrong state.")

## 7. Is decay visible?

Split the trace in half and classify each half with the matched filter. A |1⟩ shot that decays mid-readout looks
excited in the first half and ground in the second. Noise alone makes both orders equally often; decay makes only the
first. So the **asymmetry** between "excited then ground" and "ground then excited" is the decay signal, compared here
with a Gaussian null simulated at the measured noise.

In [ ]:
A_, B_ = slice(0, T // 2), slice(T // 2, T)
a0, b0, a1, b1 = mf(ev0, A_), mf(ev0, B_), mf(ev1, A_), mf(ev1, B_)
norm = lambda x, r0_, r1_: (x - 0.5 * (r0_.mean() + r1_.mean())) / (0.5 * abs(r1_.mean() - r0_.mean()))
A1n, B1n = norm(a1, a0, a1), norm(b1, b0, b1)
if a1.mean() < a0.mean():
    A1n = -A1n
if b1.mean() < b0.mean():
    B1n = -B1n
exA, exB = A1n > 0, B1n > 0
decay, rise = np.mean(exA & ~exB), np.mean(~exA & exB)
print(f"|1> shots: excited->ground {100 * decay:.3f}%, ground->excited {100 * rise:.3f}%, asymmetry {100 * (decay - rise):+.3f} points")

rng = np.random.default_rng(0)
nA, nB = np.std(A1n), np.std(B1n)
null = np.array([[np.mean((sa > 0) & (sb <= 0)) - np.mean((sa <= 0) & (sb > 0))
                  for sa, sb in [(A1n.mean() + rng.normal(0, nA, len(A1n)), B1n.mean() + rng.normal(0, nB, len(B1n)))]][0]
                 for _ in range(200)])
z = ((decay - rise) - null.mean()) / null.std()
print(f"Gaussian null asymmetry {100 * null.mean():+.3f} +/- {100 * null.std():.3f} points; excess z = {z:+.2f}")

**Reading the result.** On our run the excess asymmetry is about $z = 2.4$: suggestive, but below a 3σ threshold, so
half-to-half flips are mostly noise. Decay and preparation error still shape the classes. They skew the real
matched-filter scores toward each other (notebook 06) and leave the measured ceiling below the Gaussian prediction
(section 6 above). A model with Gaussian classes misses both.

**Things to try:** split the trace into thirds or quarters instead of halves, or repeat on the test file
(`0528_X_test_0_770.npy`).